In [ ]:
"""
Covariance Analysis Notebook

This notebook analyzes cross-covariance matrices between V1 and V4 neurons 
across different contrast levels.
"""

import os
import sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
import copy
import pickle
from tqdm import tqdm

# Add project root to Python path
# Assuming the notebook is in the Analysis directory
project_root = os.path.dirname(os.getcwd())
sys.path.append(project_root)
print(f"Project root: {project_root}")

from Utils.Create_weight_matrices import setup_parameters
from Models.Model import RingModel
from Utils.Communication import Calculate_Covariance_mean
from Utils.Coherence import create_L_matrix, create_S_matrix
from Utils.matrix_spectrum import matrix_solution

print("✓ All imports successful!") 

: 

## Configuration

Specify the path to your configuration file below.


In [ ]:
# ========== Configuration ==========
# Specify the path to your config file here
config_file = '../Results_5/config_35/config_35.yaml'  # Change this to your config file path

print(f"Config file: {config_file}")
print(f"Absolute path: {os.path.abspath(config_file)}")

# Verify the file exists
if os.path.exists(config_file):
    print(f"✓ Config file found!")
else:
    print(f"✗ Config file NOT found. Please check the path.")
    raise FileNotFoundError(f"Config file not found: {config_file}")

# Load configuration
with open(config_file, 'r') as f:
    config = yaml.safe_load(f)

# Get output directory from config file path
output_dir = os.path.dirname(config_file)
print(f"Output directory: {output_dir}")
print(f"✓ Configuration loaded successfully!")


## Model Setup

Initialize the model parameters and define neuron indices.


In [ ]:
# ========== Initialize Model Parameters ==========
params = setup_parameters(
    config=config,
    tau=1e-3,
    tauPlus=1e-3,
    N=36
)

# Initialize model
Ring_Model = RingModel(params, simulate_firing_rates=True)

# Set up indices for different neuron populations
N = params['N']
y1_idx = np.arange(0, N)  # y1 (V1 neurons, base)
N1_y = np.arange(N, 2*N)  # y1Plus (V1 neurons, firing rate)
N2_y = np.arange(3*N, 4*N)  # y2Plus (V2/V4 neurons, firing rate)

# Indices for normalization signals in the state vector
p1Plus_idx = np.arange(9*N, 10*N)  # V1 normalization signal
p4Plus_idx = np.arange(11*N, 12*N)  # V4 normalization signal

# Communication parameters
com_params = {
    'bw_y1_y4': False
}

# Get contrast and gamma values from config
contrast_vals = config['Communication']['Feedback_gain']['c_vals']
gamma_vals = config['Communication']['Feedback_gain']['gamma_vals']

print(f"✓ Model initialized successfully!")
print(f"  Number of neurons (N): {N}")
print(f"  Number of contrasts: {len(contrast_vals)}")
print(f"  Gamma values: {gamma_vals}")
print(f"  Contrast range: {min(contrast_vals)} to {max(contrast_vals)}")


## Calculate Covariance Matrices

Calculate covariance matrices for all contrast values.


In [ ]:
# ========== Calculate Covariance Matrices ==========
# Store all covariance matrices and steady states
all_cov_matrices = []
all_steady_states = []

print("\nCalculating covariance matrices for all contrasts...")
for contrast in contrast_vals:
    print(f"Processing contrast = {contrast}")
    
    # Calculate covariance matrix
    Py, ss = Calculate_Covariance_mean(
        Ring_Model,
        gamma_vals,
        contrast,
        fb_gain=True,
        input_gain_beta1=False,
        input_gain_beta4=False,
        method='RK45',
        com_params=com_params,
        delta_tau=config['noise_params']['delta_tau'] * Ring_Model.params['tau'],
        noise_potential=config['noise_params']['noise_potential'],
        noise_firing_rate=config['noise_params']['noise_firing_rate'],
        GR_noise=config['noise_params']['GR_noise'],
        t_span=[0, 10]
    )
    
    # Extract V1-V4 cross-covariance matrix
    V1_V2_cov = Py[N1_y][:, N2_y]
    all_cov_matrices.append(V1_V2_cov)
    
    # Store steady state
    all_steady_states.append(ss)

print(f"\n✓ Calculated {len(all_cov_matrices)} covariance matrices!")


## Plot 1: V1-V4 Cross-Covariance Matrices

Visualize the cross-covariance matrices between V1 and V4 neurons for each contrast.


In [ ]:
# ========== Plot Cross-Covariance Matrices ==========
# Create figure with subplots
n_contrasts = len(contrast_vals)
n_cols = 3
n_rows = int(np.ceil(n_contrasts / n_cols))

fig = plt.figure(figsize=(5*n_cols, 4*n_rows))
gs = GridSpec(n_rows, n_cols + 1, figure=fig, width_ratios=[1]*n_cols + [0.05])

# Plot each covariance matrix
for idx, (contrast, V1_V2_cov) in enumerate(zip(contrast_vals, all_cov_matrices)):
    row = idx // n_cols
    col = idx % n_cols
    
    ax = fig.add_subplot(gs[row, col])
    
    vmin, vmax = np.percentile(V1_V2_cov, [0, 100])  # exclude extreme values
    # Plot covariance matrix
    im = ax.imshow(V1_V2_cov, cmap='coolwarm', vmin=vmin, vmax=vmax,
                   aspect='auto')
    
    # Add title and labels
    ax.set_title(f'Contrast = {contrast}', fontsize=12, fontweight='bold')
    ax.set_xlabel('V1 Neuron Index', fontsize=10)
    ax.set_ylabel('V2 Neuron Index', fontsize=10)
    
    # Print statistics for this contrast
    triu_indices = np.triu_indices_from(V1_V2_cov, k=1)
    print(f"\nContrast = {contrast}:")
    print(f"  Mean covariance: {np.mean(V1_V2_cov[triu_indices]):.3e}")
    print(f"  Std covariance: {np.std(V1_V2_cov[triu_indices]):.3e}")
    print(f"  Range: [{np.min(V1_V2_cov[triu_indices]):.3e}, {np.max(V1_V2_cov[triu_indices]):.3e}]")

# Add a single colorbar for all subplots
cbar_ax = fig.add_subplot(gs[:, -1])
cbar = fig.colorbar(im, cax=cbar_ax, format='%.1e')
cbar.set_label('Covariance', fontsize=12, fontweight='bold')

# Add main title
fig.suptitle(f'V1 Covariance Matrices Across Contrast Values (γ={gamma_vals[0]})', 
             fontsize=14, fontweight='bold', y=0.98)

# Adjust layout to prevent overlap
plt.tight_layout(rect=[0, 0, 0.96, 0.96])

# Save plot
output_file = os.path.join(output_dir, 'V1_V4_cross_covariance_matrices_contrast.pdf')
plt.savefig(output_file, dpi=400, bbox_inches='tight')
print(f"\n✓ Saved cross-covariance plot to: {output_file}")
plt.show()


## Plot 2: Normalization Signals vs Contrast

Plot V1 normalization signals (p1Plus) as a function of contrast.


In [ ]:
# ========== Plot Normalization Signals ==========
print("\nPlotting normalization signals (p1Plus and p4Plus)...")

# Collect steady state values for p1Plus and p4Plus at center neuron
p1Plus_vals = []
p4Plus_vals = []

for idx, (contrast, ss) in enumerate(zip(contrast_vals, all_steady_states)):
    # Extract normalization signals from steady state (center neuron)
    p1Plus_val = ss[p1Plus_idx][N//2]  # V1 normalization signal at center
    p4Plus_val = ss[p4Plus_idx][N//2]  # V4 normalization signal at center
    
    p1Plus_vals.append(p1Plus_val)
    p4Plus_vals.append(p4Plus_val)
    
    # Print statistics
    print(f"\nContrast = {contrast}:")
    print(f"  V1 (p1Plus) center: {p1Plus_val:.3e}")
    print(f"  V4 (p4Plus) center: {p4Plus_val:.3e}")

# Create a new figure for normalization signals
fig2, ax = plt.subplots(1, 1, figsize=(10, 6))

# Plot both normalization signals vs contrast
ax.plot(contrast_vals, p1Plus_vals, 'bo-', linewidth=2, markersize=8, 
        label='V1 (p1Plus)', alpha=0.8)

# Formatting
ax.set_xlabel('Contrast', fontsize=12, fontweight='bold')
ax.set_ylabel('Normalization Signal (a)', fontsize=12, fontweight='bold')
ax.set_title(f'Normalization Signals vs Contrast (γ={gamma_vals[0]})', 
             fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')  # Log scale for contrast since values span orders of magnitude
ax.set_yscale('linear')

# Adjust layout
plt.tight_layout()

# Save normalization plot
norm_output_file = os.path.join(output_dir, 'normalization_signals_vs_contrast.pdf')
plt.savefig(norm_output_file, dpi=400, bbox_inches='tight')
print(f"\n✓ Saved normalization signals vs contrast plot to: {norm_output_file}")
plt.show()


## Plot 3: Steady State Values of y1Plus and Ratio

Plot V1 firing rates (y1Plus) and their ratio to normalization signals vs contrast.


In [ ]:
# ========== Plot Steady State Values of y1Plus ==========
print("\nPlotting steady state values of y1plus with contrast...")

# Collect steady state values for y1plus at center neuron
y1Plus_vals = []
y1Plus_std_vals = []

for idx, (contrast, ss) in enumerate(zip(contrast_vals, all_steady_states)):
    # Extract y1plus from steady state (center neuron)
    y1Plus_val = ss[N1_y][N//2]
    y1Plus_vals.append(y1Plus_val)
    y1Plus_std_vals.append(np.std(ss[y1_idx]))

# Create a new figure for steady state values of y1plus
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.plot(contrast_vals, y1Plus_vals, 'o-', color='blue', linewidth=2, markersize=8, 
        label='V1 (y1Plus)', alpha=0.8)
ax.plot(contrast_vals, np.array(y1Plus_vals)/np.array(p1Plus_vals), 'o-', color='green', linewidth=2, markersize=8, 
        label='V1 (y1Plus) / V1 (a1Plus)', alpha=0.8)
ax.set_xlabel('Contrast', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean and Ratio of Steady State Values of y1Plus and a1Plus', fontsize=12, fontweight='bold')
ax.set_title(f'Mean and Ratio of Steady State Values of y1Plus and a1Plus vs Contrast (γ={gamma_vals[0]})', 
             fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')  # Log scale for contrast
ax.set_yscale('linear')

# Save steady state values of y1plus plot
y1Plus_output_file = os.path.join(output_dir, 'steady_state_values_y1plus_a1plus_contrast.pdf')
plt.savefig(y1Plus_output_file, dpi=400, bbox_inches='tight')
print(f"\n✓ Saved steady state values of y1plus and a1plus plot to: {y1Plus_output_file}")
plt.show()


## Plot 4: Eigenvalue Spectra

Calculate and plot eigenvalue spectra of cross-covariance matrices.


In [ ]:
# ========== Plot Eigenvalue Spectra ==========
print("\nCalculating eigenvalues of covariance matrices...")

# Create a new figure for eigenvalue spectra
fig3, ax = plt.subplots(1, 1, figsize=(10, 6))

# Define colors for different contrasts (using a colormap)
colors = plt.cm.viridis(np.linspace(0, 1, len(contrast_vals)))

# Store effective dimensions and ranks for later plotting
effective_dims_95 = []
numerical_ranks = []

# Calculate and plot eigenvalues for each contrast
for idx, (contrast, V1_V2_cov) in enumerate(zip(contrast_vals, all_cov_matrices)):
    # Calculate eigenvalues
    eigenvalues = np.linalg.eigvals(V1_V2_cov)  # For cross-covariance, use C*C^T
    
    # Sort eigenvalues in descending order
    eigenvalues_sorted = np.sort(np.abs(eigenvalues))[::-1]
    
    # Calculate cumulative sum (cumulative variance explained)
    cumsum_eigenvalues = np.cumsum(eigenvalues_sorted)
    total_variance = cumsum_eigenvalues[-1]
    cumsum_normalized = cumsum_eigenvalues / total_variance
    
    # Calculate effective dimension using threshold approach
    threshold_95 = 0.95
    
    # Find the first dimension where cumulative variance exceeds the threshold
    dim_95 = next((dim for dim, var_explained in enumerate(cumsum_normalized, start=1) 
                  if var_explained >= threshold_95), len(eigenvalues_sorted))
    
    effective_dims_95.append(dim_95)
    
    # Compute Rank of Covariance Matrix
    # Method 1: Numerical rank using numpy's built-in (uses SVD with default tolerance)
    numerical_rank = np.linalg.matrix_rank(V1_V2_cov)
    numerical_ranks.append(numerical_rank)
    
    # Plot eigenvalues
    ax.plot(range(1, len(eigenvalues_sorted) + 1), eigenvalues_sorted, 
            'o-', color=colors[idx], linewidth=2, markersize=4,
            label=f'c = {contrast}', alpha=0.8)
    
    # Print statistics
    print(f"\nContrast = {contrast}:")
    print(f"  Max eigenvalue: {eigenvalues_sorted[0]:.3e}")
    print(f"  Total variance: {total_variance:.3e}")
    print(f"  Numerical rank (np.linalg.matrix_rank): {numerical_rank}")
    print(f"  Effective dimension (95% variance): {dim_95}")
    print(f"  Variance explained by top 5 dims: {cumsum_normalized[4]:.3%}")

# Formatting
ax.set_xlabel('Eigenvalue Index (sorted)', fontsize=12, fontweight='bold')
ax.set_ylabel('Eigenvalue Magnitude', fontsize=12, fontweight='bold')
ax.set_title(f'Eigenvalue Spectra of Cross-Covariance Matrices (γ={gamma_vals[0]})', 
             fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10, ncol=2)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')  # Log scale to see the decay better

# Adjust layout
plt.tight_layout()

# Save eigenvalue plot
eigen_output_file = os.path.join(output_dir, 'eigenvalue_spectra_contrast.pdf')
plt.savefig(eigen_output_file, dpi=400, bbox_inches='tight')
print(f"\n✓ Saved eigenvalue spectra plot to: {eigen_output_file}")
plt.show()


## Plot 5: Rank and Effective Dimensionality vs Contrast

Plot numerical rank and effective dimensionality as a function of contrast.


In [ ]:
# ========== Plot Rank and Effective Dimensionality vs Contrast ==========
print("\nPlotting rank and effective dimensionality vs contrast...")

# Create a figure with subplots
fig4, ax = plt.subplots(1, 1, figsize=(10, 10))

# Plot: Different rank measures
ax.plot(contrast_vals, numerical_ranks, 'ko-', linewidth=2, markersize=8, 
        label='Numerical rank (SVD)', alpha=0.8)
ax.plot(contrast_vals, effective_dims_95, 'mo-', linewidth=2, markersize=8, 
        label='Effective dim (95% variance)', alpha=0.8)

ax.set_xlabel('Contrast', fontsize=12, fontweight='bold')
ax.set_ylabel('Dimensionality/Rank ', fontsize=12, fontweight='bold')
ax.set_title(f'Dimensionality/Rank vs Contrast (γ={gamma_vals[0]})', 
             fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')  # Log scale for contrast

# Adjust layout
plt.tight_layout()

# Save dimensionality plot
dim_output_file = os.path.join(output_dir, 'rank_dimensionality_contrast.pdf')
plt.savefig(dim_output_file, dpi=400, bbox_inches='tight')
print(f"\n✓ Saved rank and dimensionality plot to: {dim_output_file}")
plt.show()


## Analysis Complete! 🎉

All plots have been generated and saved to the output directory.


In [ ]:
# ========== Summary ==========
print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print(f"\nNumber of V1 neurons: {len(N1_y)}")
print(f"Number of V2/V4 neurons: {len(N2_y)}")
print(f"Gamma value: {gamma_vals[0]}")
print(f"Number of contrasts analyzed: {len(contrast_vals)}")
print(f"\nAll plots saved to: {output_dir}")
print("\nGenerated files:")
print("  1. V1_V4_cross_covariance_matrices_contrast.pdf")
print("  2. normalization_signals_vs_contrast.pdf")
print("  3. steady_state_values_y1plus_a1plus_contrast.pdf")
print("  4. eigenvalue_spectra_contrast.pdf")
print("  5. rank_dimensionality_contrast.pdf")
print("\n" + "="*80)
